# NRR activity ML: limiting potential with the fewest features, and free energies from base features
Post-midsem extension of mid-sem Models 1–3, on a larger merged dataset (S/N-graphene SACs, Xu 2025 + carbon-nitride SACs, Xu 2024).

**Rules used throughout**
- Test set = 20 % of catalysts, held out by SAC (end-on and side-on rows of one SAC never split across train/test). It is used once, at the end.
- All tuning and feature selection use 5-fold GroupKFold (by SAC) on the training set only.
- U_L = −max(ΔG*N2→*N2H, ΔG*NH2→*NH3): proton–electron steps only (N2 adsorption is not an electrochemical step).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import re, numpy as np, pandas as pd, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from mendeleev import element
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error as MAE, r2_score as R2
from xgboost import XGBRegressor
import shap
DATA='./'; FAST=False; SEED=42
FAM_COL={'SxNy':'#4c72b0','CxNy':'#dd8452'}; FAM_LAB={'SxNy':'S/N-graphene','CxNy':'carbon nitride'}

## 1. Dataset
`merged_SAC_data.csv` (302 rows) plus the 43 S/N SACs that have complete end-on data but no side-on data (Xu 2025 Tables S1/S3), which the mid-sem data left out.

In [2]:
m=pd.read_csv(DATA+'merged_SAC_data.csv')
m['Family']=np.where(m.Support_family=='CxNy','CxNy','SxNy')
s1=pd.read_csv(DATA+'Table_S1.csv'); s1.columns=['SAC','Ebin','G_end','G_side','GH_end','GH_side','GNH3']
s3=pd.read_csv(DATA+'Table_S3.csv'); s3.columns=['SAC','bl_end','bl_side','E_end','E_side','EH_end','EH_side','E_NH2','phi']
x=s1.merge(s3,on='SAC').dropna(subset=['G_end','GH_end','GNH3','bl_end','E_end','EH_end','E_NH2'])
x=x[~x.SAC.isin(m.SAC)]
ext=pd.DataFrame({'SAC':x.SAC,'bond_length':x.bl_end,'Eads_N2H':x.EH_end,'Ebin':x.Ebin,'ΔG*N2':x.G_end,'ΔG*N2-N2H':x.GH_end,'ΔG*NH2-NH3':x.GNH3,
                  'mode':0,'Metal':x.SAC.str.split('@').str[0],'Support':x.SAC.str.split('@').str[1],'Family':'SxNy','Ebin_source':'Xu2025 Table S1 (exact)'})
cols=['SAC','Metal','Support','Family','mode','bond_length','Eads_N2H','Ebin','ΔG*N2','ΔG*N2-N2H','ΔG*NH2-NH3','Ebin_source','stable_stage1']
d=pd.concat([m[cols],ext],ignore_index=True)
d['UL']=-d[['ΔG*N2-N2H','ΔG*NH2-NH3']].max(axis=1)
# known typo in Xu 2025 Table S1: W@S2N2-HEX end-on ΔG*N2 = +0.888 while its ΔE*N2 = −1.000 -> excluded from the ΔG*N2 model only
d.loc[(d.SAC=='W@S2N2-HEX')&(d['mode']==0),'ΔG*N2']=np.nan
# ---- base features: tabulated metal properties + site descriptors + adsorption mode (no DFT)
ref=pd.read_csv(DATA+'stability_reference_values.csv',index_col='Metal')
def nd(mm):
    dd=[(n,c) for (n,l),c in element(mm).ec.conf.items() if l=='d']; return max(dd)[1] if dd else 0
P={}
for mm in d.Metal.unique():
    e=element(mm); P[mm]=dict(Z=e.atomic_number,period=e.period,group=e.group_id,Nd=nd(mm),chi_M=e.en_pauling,IE1=e.ionenergies.get(1),
                             EA=(e.electron_affinity or 0.0),r_atom=e.atomic_radius,Ecoh=float(ref.loc[mm,'Ecoh_eV']))
d=d.join(pd.DataFrame(P).T.astype(float),on='Metal')
CHI={'C':2.55,'N':3.04,'S':2.58}
def site(s):
    if s.startswith('g-'):
        a,b=re.findall(r'C(\d+)N(\d*)',s)[0]; xx,yy=int(a),int(b or 1)
        return dict(n_S=0,N_frac=yy/(xx+yy),chi_site=(xx*CHI['C']+yy*CHI['N'])/(xx+yy),is_CN=1)
    nS=int(s[1]) if s.startswith('S') else 0
    return dict(n_S=nS,N_frac=(4-nS)/4,chi_site=(nS*CHI['S']+(4-nS)*CHI['N'])/4,is_CN=0)
d=d.join(pd.DataFrame({s:site(s) for s in d.Support.unique()}).T.astype(float),on='Support')
d['EN_diff']=d.chi_site-d.chi_M
BASE=['Z','period','group','Nd','chi_M','IE1','EA','r_atom','Ecoh','n_S','N_frac','chi_site','is_CN','EN_diff','mode']
TIER={**{f:'no DFT' for f in BASE},'Ebin':'support DFT','bond_length':'*N2 DFT','Eads_N2H':'*N2H DFT'}
BASE_E=BASE+['Ebin']                     # "base features": everything except adsorbate DFT (Ebin is already computed in stage 1)
d=d.reset_index(drop=True)
print('rows:',len(d),'| SACs:',d.SAC.nunique(),'| by family:',d.groupby('Family').SAC.nunique().to_dict(),'| features in base set:',len(BASE_E))
d.to_csv(DATA+'activity_ml_dataset.csv',index=False)

rows: 345 | SACs: 254 | by family: {'CxNy': 120, 'SxNy': 134} | features in base set: 16


## 2. Train/test split by catalyst

In [3]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
tr,te=next(gss.split(d,groups=d.SAC)); TR,TE=d.iloc[tr].reset_index(drop=True),d.iloc[te].reset_index(drop=True)
print('train rows',len(TR),'SACs',TR.SAC.nunique(),'| test rows',len(TE),'SACs',TE.SAC.nunique(),'| shared SACs:',len(set(TR.SAC)&set(TE.SAC)))
print('test by family:',TE.groupby('Family').SAC.nunique().to_dict())
gkf=GroupKFold(n_splits=5)

train rows 274 SACs 203 | test rows 71 SACs 51 | shared SACs: 0
test by family: {'CxNy': 23, 'SxNy': 28}


## 3. Models and tuning (KRR, RFR, GBR, XGBR — as in Das et al.; grid search with grouped 5-fold CV on train)

In [4]:
def grids():
    g={'KRR':(make_pipeline(StandardScaler(),KernelRidge()),{'kernelridge__kernel':['rbf','laplacian'],'kernelridge__alpha':[1e-3,1e-2,1e-1,1],'kernelridge__gamma':[0.01,0.03,0.1,0.3]}),
       'RFR':(RandomForestRegressor(n_estimators=300,random_state=0,n_jobs=2),{'max_depth':[None,8],'min_samples_leaf':[1,3],'max_features':[0.5,1.0]}),
       'GBR':(GradientBoostingRegressor(random_state=0,subsample=0.8),{'n_estimators':[300,600],'learning_rate':[0.05,0.1],'max_depth':[2,3]}),
       'XGBR':(XGBRegressor(random_state=0,verbosity=0,n_jobs=2,subsample=0.8),{'n_estimators':[300,600],'learning_rate':[0.05,0.1],'max_depth':[2,3,4]})}
    if FAST: g={k:(v[0],{kk:vv[:1] for kk,vv in v[1].items()}) for k,v in g.items()}
    return g
def run(target,feats,models=('KRR','RFR','GBR','XGBR')):
    a=TR.dropna(subset=[target]); b=TE.dropna(subset=[target]); rows=[]; fitted={}
    for name in models:
        est,grid=grids()[name]
        gs=GridSearchCV(est,grid,cv=gkf,scoring='neg_mean_absolute_error').fit(a[feats],a[target],groups=a.SAC)
        mdl=gs.best_estimator_; pa=mdl.predict(a[feats]); pb=mdl.predict(b[feats])
        rows.append(dict(target=target,model=name,n_feat=len(feats),train_MAE=MAE(a[target],pa),CV_MAE=-gs.best_score_,test_MAE=MAE(b[target],pb),test_R2=R2(b[target],pb)))
        fitted[name]=(mdl,pb)
    t=pd.DataFrame(rows).round(3); best=t.loc[t.CV_MAE.idxmin(),'model']     # model chosen on CV, not on test
    return t,fitted,best

## 4. Free energies of the key steps from base features (mid-sem Model 2, now on ΔG directly)

In [5]:
DG=['ΔG*N2','ΔG*N2-N2H','ΔG*NH2-NH3']; resG={}; tabs=[]
for t in DG:
    tab,fit,best=run(t,BASE_E); resG[t]=(fit,best); tab['features']='base (no adsorbate DFT)'; tabs.append(tab)
    tab2,fit2,best2=run(t,BASE_E+['bond_length'],models=('KRR','XGBR')); tab2['features']='base + N≡N'; tabs.append(tab2); resG[t+' +NN']=(fit2,best2)
TG=pd.concat(tabs); print(TG.to_string(index=False))

    target model  n_feat  train_MAE  CV_MAE  test_MAE  test_R2                features
     ΔG*N2   KRR      16      0.068   0.180     0.150    0.732 base (no adsorbate DFT)
     ΔG*N2   RFR      16      0.070   0.192     0.186    0.647 base (no adsorbate DFT)
     ΔG*N2   GBR      16      0.081   0.183     0.167    0.686 base (no adsorbate DFT)
     ΔG*N2  XGBR      16      0.028   0.183     0.163    0.660 base (no adsorbate DFT)
     ΔG*N2   KRR      17      0.122   0.166     0.154    0.740              base + N≡N
     ΔG*N2  XGBR      17      0.016   0.170     0.175    0.654              base + N≡N
 ΔG*N2-N2H   KRR      16      0.085   0.205     0.189    0.595 base (no adsorbate DFT)
 ΔG*N2-N2H   RFR      16      0.076   0.210     0.217    0.486 base (no adsorbate DFT)
 ΔG*N2-N2H   GBR      16      0.120   0.213     0.223    0.470 base (no adsorbate DFT)
 ΔG*N2-N2H  XGBR      16      0.064   0.214     0.199    0.533 base (no adsorbate DFT)
 ΔG*N2-N2H   KRR      17      0.054   0.186

In [6]:
fig,axs=plt.subplots(1,3,figsize=(13,4.2))
for ax,t in zip(axs,DG):
    fit,best=resG[t]; b=TE.dropna(subset=[t]); p=fit[best][1]
    for f in ['SxNy','CxNy']:
        k=(b.Family==f).values; ax.scatter(b[t].values[k],p[k],s=14,c=FAM_COL[f],label=FAM_LAB[f])
    lo,hi=min(b[t].min(),p.min())-0.1,max(b[t].max(),p.max())+0.1; ax.plot([lo,hi],[lo,hi],'k--',lw=1)
    ax.set_title(f'{t} — {best}: test MAE {MAE(b[t],p):.2f} eV, R² {R2(b[t],p):.2f}',fontsize=10); ax.set_xlabel('DFT (eV)'); ax.set_ylabel('predicted (eV)')
axs[0].legend(fontsize=8); fig.suptitle('Free energies from base features (held-out test catalysts)'); plt.tight_layout(); plt.savefig(DATA+'fig1_dG_parity_test.png',dpi=200)

### Which features drive each free energy? (SHAP on the tuned XGBR model, base features)

In [7]:
fig,axs=plt.subplots(1,3,figsize=(13,4.5)); SH={}
for ax,t in zip(axs,DG):
    mdl=resG[t][0]['XGBR'][0]; a=TR.dropna(subset=[t])
    sv=shap.TreeExplainer(mdl).shap_values(a[BASE_E]); imp=pd.Series(np.abs(sv).mean(0),index=BASE_E).sort_values()
    SH[t]=imp[::-1]; imp.tail(10).plot.barh(ax=ax,color='#4c72b0'); ax.set_title(t,fontsize=10); ax.set_xlabel('mean |SHAP| (eV)')
fig.suptitle('Feature contributions to the free energies (top 10)'); plt.tight_layout(); plt.savefig(DATA+'fig2_dG_shap.png',dpi=200)
print(pd.DataFrame({t:[f'{i} ({v:.3f})' for i,v in SH[t].head(6).items()] for t in DG}).to_string(index=False))

          ΔG*N2       ΔG*N2-N2H    ΔG*NH2-NH3
   Ecoh (0.240)    Ecoh (0.206) group (0.312)
   mode (0.103)   group (0.123)  Ecoh (0.154)
   Ebin (0.094)    mode (0.077)    Nd (0.126)
EN_diff (0.051)    Ebin (0.067)  Ebin (0.099)
  group (0.048) EN_diff (0.052) is_CN (0.092)
    n_S (0.031)  N_frac (0.036)     Z (0.081)


## 5. Limiting potential with the fewest features
Forward selection (KRR, grouped 5-fold CV on train only). Two candidate pools:
- **P1, no adsorbate DFT**: base features (metal, site, mode) + Ebin
- **P2, + cheap adsorbate DFT**: P1 + N≡N bond length (*N2) + ΔE_ads(N2H)

The smallest set whose CV MAE is within one standard error of the best is chosen (1-SE rule).

In [8]:
krr_sel=make_pipeline(StandardScaler(),KernelRidge(kernel='rbf',alpha=0.03,gamma=0.1))
def cv_mae(feats):
    e=[]
    for i,j in gkf.split(TR,groups=TR.SAC):
        mdl=krr_sel.fit(TR.iloc[i][feats],TR.iloc[i].UL); e.append(MAE(TR.iloc[j].UL,mdl.predict(TR.iloc[j][feats])))
    return np.mean(e),np.std(e)/np.sqrt(len(e))
def forward(pool,kmax=10):
    sel=[];path=[];rem=list(pool)
    while rem and len(sel)<kmax:
        sc=[(cv_mae(sel+[f]),f) for f in rem]; (mu,se),f=min(sc,key=lambda z:z[0][0]); sel.append(f); rem.remove(f); path.append((len(sel),f,mu,se))
    return pd.DataFrame(path,columns=['k','added','CV_MAE','SE'])
P1=BASE_E; P2=BASE_E+['bond_length','Eads_N2H']
path1=forward(P1,12 if not FAST else 4); path2=forward(P2,12 if not FAST else 4)
def one_se(p):
    i=p.CV_MAE.idxmin(); thr=p.CV_MAE[i]+p.SE[i]; return int(p[p.CV_MAE<=thr].k.min())
k1,k2=one_se(path1),one_se(path2)
for nm,p,k in [('P1 (no adsorbate DFT)',path1,k1),('P2 (+ N≡N, ΔE_ads N2H)',path2,k2)]:
    p['tier']=p.added.map(TIER); print(nm,'-> chosen k =',k); print(p.round(3).to_string(index=False),'\n')
fig,ax=plt.subplots(figsize=(7,4))
for p,k,lab,c in [(path1,k1,'P1: no adsorbate DFT','#4c72b0'),(path2,k2,'P2: + N≡N and ΔE_ads(N2H)','#c44e52')]:
    ax.errorbar(p.k,p.CV_MAE,yerr=p.SE,marker='o',ms=4,capsize=2,color=c,label=lab); ax.scatter([k],[p.set_index('k').CV_MAE[k]],s=140,facecolors='none',edgecolors=c,linewidths=2)
ax.set_xlabel('number of features (forward selection)'); ax.set_ylabel('grouped 5-fold CV MAE of U_L (V)'); ax.set_title('Fewest features for U_L (circle = chosen by 1-SE rule)')
ax.legend(); plt.tight_layout(); plt.savefig(DATA+'fig3_UL_feature_selection.png',dpi=200)
F1=list(path1.added[:k1]); F2=list(path2.added[:k2]); print('minimal P1:',F1,'\nminimal P2:',F2)

P1 (no adsorbate DFT) -> chosen k = 5
 k   added  CV_MAE    SE        tier
 1      Nd   0.264 0.010      no DFT
 2   chi_M   0.248 0.012      no DFT
 3    mode   0.242 0.013      no DFT
 4    Ecoh   0.227 0.013      no DFT
 5   is_CN   0.207 0.007      no DFT
 6    Ebin   0.200 0.009 support DFT
 7  r_atom   0.201 0.008      no DFT
 8   group   0.206 0.008      no DFT
 9  period   0.210 0.009      no DFT
10     IE1   0.214 0.009      no DFT
11 EN_diff   0.217 0.013      no DFT
12       Z   0.220 0.012      no DFT 

P2 (+ N≡N, ΔE_ads N2H) -> chosen k = 6
 k       added  CV_MAE    SE     tier
 1    Eads_N2H   0.207 0.010 *N2H DFT
 2        mode   0.178 0.011   no DFT
 3       is_CN   0.172 0.010   no DFT
 4       chi_M   0.161 0.010   no DFT
 5          Nd   0.156 0.009   no DFT
 6          EA   0.152 0.007   no DFT
 7 bond_length   0.149 0.006  *N2 DFT
 8        Ecoh   0.150 0.005   no DFT
 9       group   0.151 0.005   no DFT
10           Z   0.156 0.005   no DFT
11         IE1   0.158

### Final U_L models: tuned on train, scored once on the test set

In [9]:
TU=[]
tabA,fitA,bestA=run('UL',BASE_E); tabA['features']=f'all base ({len(BASE_E)})'; TU.append(tabA)
tab1,fit1,best1=run('UL',F1); tab1['features']=f'minimal P1 ({len(F1)})'; TU.append(tab1)
tab2,fit2,best2=run('UL',F2); tab2['features']=f'minimal P2 ({len(F2)})'; TU.append(tab2)
# U_L assembled from the predicted free energies (error propagation, mid-sem point)
pN2H=resG['ΔG*N2-N2H'][0][resG['ΔG*N2-N2H'][1]][1]; pNH3=resG['ΔG*NH2-NH3'][0][resG['ΔG*NH2-NH3'][1]][1]
ul_from_dg=-np.maximum(pN2H,pNH3)
TU.append(pd.DataFrame([dict(target='UL',model='from predicted ΔG',n_feat=len(BASE_E),train_MAE=np.nan,CV_MAE=np.nan,test_MAE=MAE(TE.UL,ul_from_dg),test_R2=R2(TE.UL,ul_from_dg),features='base, via ΔG models')]).round(3))
TUL=pd.concat(TU); print(TUL.to_string(index=False))

target             model  n_feat  train_MAE  CV_MAE  test_MAE  test_R2            features
    UL               KRR      16      0.138   0.214     0.225    0.284       all base (16)
    UL               RFR      16      0.073   0.217     0.239    0.233       all base (16)
    UL               GBR      16      0.092   0.216     0.273    0.022       all base (16)
    UL              XGBR      16      0.059   0.215     0.250    0.088       all base (16)
    UL               KRR       5      0.184   0.207     0.230    0.294      minimal P1 (5)
    UL               RFR       5      0.166   0.208     0.236    0.237      minimal P1 (5)
    UL               GBR       5      0.158   0.209     0.252    0.159      minimal P1 (5)
    UL              XGBR       5      0.156   0.211     0.253    0.148      minimal P1 (5)
    UL               KRR       6      0.119   0.154     0.177    0.618      minimal P2 (6)
    UL               RFR       6      0.058   0.161     0.171    0.601      minimal P2 (6)

In [10]:
fig,axs=plt.subplots(1,4,figsize=(17,4.3))
panels=[(f'all base features, {bestA}',fitA[bestA][1]),(f'minimal P1 ({len(F1)} feat.), {best1}',fit1[best1][1]),(f'minimal P2 ({len(F2)} feat.), {best2}',fit2[best2][1]),('from predicted ΔG (base features)',ul_from_dg)]
for ax,(lab,p) in zip(axs,panels):
    for f in ['SxNy','CxNy']:
        k=(TE.Family==f).values; ax.scatter(TE.UL.values[k],p[k],s=14,c=FAM_COL[f],label=FAM_LAB[f])
    lo,hi=min(TE.UL.min(),p.min())-0.05,max(TE.UL.max(),p.max())+0.05; ax.plot([lo,hi],[lo,hi],'k--',lw=1)
    ax.set_title(f'{lab}\ntest MAE {MAE(TE.UL,p):.2f} V, R² {R2(TE.UL,p):.2f}, ρ {spearmanr(TE.UL,p)[0]:.2f}',fontsize=9); ax.set_xlabel('DFT U_L (V)'); ax.set_ylabel('predicted U_L (V)')
axs[0].legend(fontsize=8); fig.suptitle('Limiting potential on held-out test catalysts'); plt.tight_layout(); plt.savefig(DATA+'fig4_UL_parity_test.png',dpi=200)

### Which features drive U_L? (SHAP on the minimal-feature models)

In [11]:
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
for ax,(F,fit,best,title) in zip(axs,[(F1,fit1,best1,'minimal P1 (no adsorbate DFT)'),(F2,fit2,best2,'minimal P2 (+ cheap adsorbate DFT)')]):
    mdl=fit[best][0]; bg=shap.sample(TR[F],60,random_state=0)
    if best in ('RFR','GBR','XGBR'): sv=shap.TreeExplainer(mdl).shap_values(TE[F])
    else:
        f=lambda X,mdl=mdl,F=F: mdl.predict(pd.DataFrame(X,columns=F))
        sv=shap.KernelExplainer(f,bg.values).shap_values(TE[F].values,nsamples=200,silent=True)
    plt.sca(ax); shap.summary_plot(sv,TE[F],show=False,plot_size=None,color_bar=False); ax.set_title(f'{title} — {best}',fontsize=10)
fig.text(0.5,0.005,'point colour = feature value (blue = low, red = high); x = effect on predicted U_L (V); positive = less negative U_L (better)',ha='center',fontsize=9)
plt.tight_layout(rect=(0,0.04,1,1)); plt.savefig(DATA+'fig5_UL_shap.png',dpi=200)

## 6. Inference
**a) Transfer between families** (train on one family, predict the other; base features, XGBR and KRR)

In [12]:
rows=[]
for target in ['UL']+DG:
    dd=d.dropna(subset=[target])
    for trf,tef in [('SxNy','CxNy'),('CxNy','SxNy')]:
        a,b=dd[dd.Family==trf],dd[dd.Family==tef]
        for name,mdl in [('KRR',make_pipeline(StandardScaler(),KernelRidge(kernel='rbf',alpha=0.03,gamma=0.1))),('XGBR',XGBRegressor(n_estimators=400,max_depth=3,learning_rate=0.05,subsample=0.8,verbosity=0,random_state=0))]:
            p=mdl.fit(a[BASE_E],a[target]).predict(b[BASE_E]); rows.append(dict(target=target,train=FAM_LAB[trf],test=FAM_LAB[tef],model=name,MAE=MAE(b[target],p),R2=R2(b[target],p),spearman=spearmanr(b[target],p)[0]))
TT=pd.DataFrame(rows).round(3); print(TT.to_string(index=False))

    target          train           test model   MAE     R2  spearman
        UL   S/N-graphene carbon nitride   KRR 0.398 -0.176     0.331
        UL   S/N-graphene carbon nitride  XGBR 0.328  0.264     0.609
        UL carbon nitride   S/N-graphene   KRR 0.667 -3.096     0.116
        UL carbon nitride   S/N-graphene  XGBR 0.343 -0.257     0.292
     ΔG*N2   S/N-graphene carbon nitride   KRR 0.334 -0.553     0.511
     ΔG*N2   S/N-graphene carbon nitride  XGBR 0.322 -0.422     0.497
     ΔG*N2 carbon nitride   S/N-graphene   KRR 0.370 -0.183     0.132
     ΔG*N2 carbon nitride   S/N-graphene  XGBR 0.331  0.024     0.484
 ΔG*N2-N2H   S/N-graphene carbon nitride   KRR 0.371  0.009     0.555
 ΔG*N2-N2H   S/N-graphene carbon nitride  XGBR 0.254  0.591     0.790
 ΔG*N2-N2H carbon nitride   S/N-graphene   KRR 0.538 -1.070     0.319
 ΔG*N2-N2H carbon nitride   S/N-graphene  XGBR 0.297  0.351     0.629
ΔG*NH2-NH3   S/N-graphene carbon nitride   KRR 0.599 -0.124     0.629
ΔG*NH2-NH3   S/N-gra

**b) Does the model find the best catalysts?** Rank agreement and top-10 recovery on the test set; stage-1-stable test catalysts listed.

In [13]:
pbest=fit2[best2][1]; pP1=fit1[best1][1]
for lab,p in [('minimal P1',pP1),('minimal P2',pbest),('from ΔG',ul_from_dg)]:
    top_dft=set(TE.nlargest(10,'UL').index); top_ml=set(pd.Series(p).nlargest(10).index)
    print(f'{lab:12s} Spearman {spearmanr(TE.UL,p)[0]:.2f} | top-10 DFT catalysts recovered in ML top-10: {len(top_dft&top_ml)}/10')
st=TE.assign(UL_pred_P1=pP1.round(2),UL_pred_P2=pbest.round(2))
print(st[st.stable_stage1==1][['SAC','mode','UL','UL_pred_P1','UL_pred_P2']].sort_values('UL',ascending=False).to_string(index=False))
TE.assign(UL_pred_P1=pP1,UL_pred_P2=pbest,UL_from_dG=ul_from_dg).round(3).to_csv(DATA+'activity_ml_test_predictions.csv',index=False)
TG.to_csv(DATA+'activity_ml_dG_results.csv',index=False); TUL.to_csv(DATA+'activity_ml_UL_results.csv',index=False)

minimal P1   Spearman 0.63 | top-10 DFT catalysts recovered in ML top-10: 5/10
minimal P2   Spearman 0.74 | top-10 DFT catalysts recovered in ML top-10: 4/10
from ΔG      Spearman 0.69 | top-10 DFT catalysts recovered in ML top-10: 8/10
      SAC  mode     UL  UL_pred_P1  UL_pred_P2
Mo@g-C4N3     0 -0.142       -0.62       -0.58
     V@N4     1 -0.418       -0.57       -0.57
  Fe@S1N3     1 -0.736       -0.95       -1.03
    Mn@N4     1 -0.758       -0.88       -1.22
     V@N4     0 -0.884       -0.76       -0.77
Ti@g-C4N3     0 -0.938       -0.92       -1.10
Zr@g-C4N3     0 -1.067       -0.77       -0.97
  Fe@S1N3     0 -1.141       -1.13       -1.24
    Mn@N4     0 -1.218       -0.94       -1.33
Co@g-C4N3     0 -1.248       -1.23       -1.19
